# Kernel Ridge Regression

## 1. Problem Setup

You are given training data

$$
\{(x_1, y_1), (x_2, y_2), \dots, (x_N, y_N)\}
$$

where

$$
x_i \in \mathbb{R}^D, \quad y_i \in \mathbb{R}
$$

Goal: predict

$$
y(x)
$$

for a new input $x$.

---

## 2. Kernel Function

Choose a kernel such as Gaussian kernel

$$
K(x_i, x_j) = \exp\Bigg(-\frac{\|x_i - x_j\|^2}{2h^2}\Bigg)
$$

This measures similarity between two points.

---

## 3. Kernel Matrix

Construct the kernel matrix

$$
K \in \mathbb{R}^{N \times N}
$$

where

$$
K_{ij} = K(x_i, x_j)
$$

Matrix form:

$$
K =
\begin{bmatrix}
K(x_1, x_1) & K(x_1, x_2) & \dots & K(x_1, x_N) \\
K(x_2, x_1) & K(x_2, x_2) & \dots & K(x_2, x_N) \\
\vdots & \vdots & \ddots & \vdots \\
K(x_N, x_1) & K(x_N, x_2) & \dots & K(x_N, x_N)
\end{bmatrix}
$$


---

## 4. Optimization Problem

Kernel ridge regression minimizes

$$
\min_f \sum_{i=1}^N (y_i - f(x_i))^2 + \lambda \|f\|^2
$$

where $\lambda > 0$ is the regularization parameter.

---

## 5. Representer Theorem

The solution has the form

$$
f(x) = \sum_{i=1}^N \alpha_i K(x, x_i)
$$

So the unknowns become

$$
\alpha_1, \alpha_2, \dots, \alpha_N
$$

---

## 6. Matrix Form

Define

$$
\alpha =
\begin{bmatrix}
\alpha_1 \\ \alpha_2 \\ \vdots \\ \alpha_N
\end{bmatrix}
$$

Predictions on training points:

$$
f(X) = K \alpha
$$

---

## 7. Solution 

We want to minimize:

$$
J(\alpha) = \sum_{i=1}^N (y_i - f(x_i))^2 + \lambda \|f\|^2
$$

Substitute $f(x_i) = \sum_{j=1}^N \alpha_j K(x_i, x_j)$:

$$
J(\alpha) = \sum_{i=1}^N \Big(y_i - \sum_{j=1}^N \alpha_j K(x_i, x_j)\Big)^2 + \lambda \alpha^\top K \alpha
$$

In matrix form:

$$
J(\alpha) = (y - K\alpha)^\top (y - K\alpha) + \lambda \alpha^\top K \alpha
$$

To find the minimum, take derivative with respect to $\alpha$ and set to zero:

$$
\frac{\partial J}{\partial \alpha} = -2 K^\top (y - K \alpha) + 2 \lambda K \alpha = 0
$$

Since $K$ is symmetric ($K^\top = K$):

$$
-2 K (y - K \alpha) + 2 \lambda K \alpha = 0
$$

$$
-2 K y + 2 K K \alpha + 2 \lambda K \alpha = 0
$$

$$
(K K + \lambda K) \alpha = K y
$$

Factor $K$:

$$
K(K + \lambda I) \alpha = K y
$$

Multiply both sides by $K^{-1}$ (assuming $K$ is invertible after regularization):

$$
(K + \lambda I) \alpha = y
$$

Hence the solution:

$$
\alpha = (K + \lambda I)^{-1} y
$$

where $I$ is the identity matrix.

---

## 8. Prediction for Test Points

Compute kernel between test and training points

$$
K_{\text{test}}
$$

where

$$
K_{\text{test}}(i,j) = K(x_i^{\text{test}}, x_j^{\text{train}})
$$


Prediction:

$$
y_{\text{pred}} = K_{\text{test}} \alpha
$$

---



In [1]:
class KernelRidgeRegression:
    def __init__(self,lambda_=0.1, h=1):
        self.lambda_ = lambda_
        self.h = h
        self.alpha= None


    def _gaussian_kernel(self, X_test, X_train):
        D_squared = (
            np.sum(X_test**2, axis=1, keepdims=True) + np.sum(X_train**2, axis=1) - 2 * X_test @ X_train.T)
        h = max(self.h, 0.001)  
        K = np.exp(-D_squared / (2 * h**2))
        return K
        


    def fit(self,X,y):
        self.X_train = X

        self.X_train = np.asarray(self.X_train)
        y = np.asarray(y)

        y = y.reshape(-1)

        if self.X_train.ndim ==1:
            self.X_train = self.X_train.reshape(-1,1)

        N  = len(self.X_train)

        I = np.eye(N)

        K_train = self._gaussian_kernel(self.X_train,self.X_train)
        self.alpha = np.linalg.solve(K_train +  self.lambda_ * I,  y)
        print("Kernel Matrix:")
        print(K_train)
        print("Dual coefficients (alpha):")
        print(self.alpha)
        


    def predict(self,X):
        if self.alpha is None:
            raise ValueError("Model not fitted. Call fit() first.")
        X = np.asarray(X)

        if X.ndim ==1:
            X = X.reshape(-1,1)
        
        K_test = self._gaussian_kernel(X,self.X_train)
        print("Test Kernel:")
        print(K_test)


        return K_test @ self.alpha


## 1. Create a Simple Dataset

We construct a small dataset with three training points.

$$
X_{train} =
\begin{bmatrix}
0 \\
1 \\
3
\end{bmatrix}
$$

and corresponding targets

$$
y =
\begin{bmatrix}
1 \\
2 \\
4
\end{bmatrix}
$$

We evaluate predictions at the test point

$$
x_{test} = 1.5
$$

Kernel Ridge Regression combines **kernel methods** with **ridge regularization** to control model complexity.



In [2]:
import numpy as np

X_train = np.array([[0],[1],[3]])
y_train = np.array([1,2,4])

X_test = np.array([[1.5]])

## 2. Distance Matrix

We first compute the squared Euclidean distance between training points.

$$
D(x_i,x_j) = (x_i-x_j)^2
$$

This distance matrix is used to construct the kernel matrix.

In [3]:
D_train = (X_train - X_train.T)**2
print("Training Distance Matrix:")
print(D_train)

Training Distance Matrix:
[[0 1 9]
 [1 0 4]
 [9 4 0]]


## 3. Kernel Matrix

Using the Gaussian (RBF) kernel

$$
K(x_i,x_j)=
\exp
\left(
-\frac{(x_i-x_j)^2}{2h^2}
\right)
$$

The kernel matrix measures similarity between all training points.

In [4]:
h = 1.0

K = np.exp(-D_train/(2*h**2))

print("Kernel Matrix:")
print(K)

Kernel Matrix:
[[1.         0.60653066 0.011109  ]
 [0.60653066 1.         0.13533528]
 [0.011109   0.13533528 1.        ]]


## 4. Regularization

Kernel Ridge Regression adds a ridge regularization term.

The model solves

$$
(K + \lambda I)\alpha = y
$$

where

- $\lambda$ is the **regularization parameter**
- $I$ is the identity matrix
- $\alpha$ are the dual coefficients

Regularization improves numerical stability and prevents overfitting.

In [5]:
lam = 0.5

I = np.eye(len(X_train))

alpha = np.linalg.solve(K + lam*I, y_train)

print("Dual coefficients (alpha):")
print(alpha)

Dual coefficients (alpha):
[0.24193835 1.00323689 2.57435931]


## 5. Kernel Between Test and Training Points

To make predictions we compute the kernel between the test point and the training points

$$
K(x,x_i)=
\exp
\left(
-\frac{(x-x_i)^2}{2h^2}
\right)
$$

In [6]:
D_test = (X_test - X_train.T)**2
K_test = np.exp(-D_test/(2*h**2))

print("Test Kernel Vector:")
print(K_test)

Test Kernel Vector:
[[0.32465247 0.8824969  0.32465247]]


## 6. Prediction

Kernel Ridge Regression predicts using

$$
\hat{y}(x) =
\sum_{i=1}^{N} \alpha_i K(x,x_i)
$$

or in matrix form

$$
\hat{y} = K_{test} \alpha
$$

In [7]:
y_pred = K_test @ alpha

print("Prediction at x = 1.5:")
print(y_pred)

Prediction at x = 1.5:
[1.79967143]


## 7. Effect of Regularization

Small $\lambda$:

- Model closely fits the training data
- Risk of overfitting

Large $\lambda$:

- Coefficients shrink
- Predictions become smoother

Thus $\lambda$ controls the **bias–variance trade-off** similarly to ridge regression.

In [8]:
model = KernelRidgeRegression(lambda_=.5, h=1)
model.fit(X_train,y_train)
print('\n')
model.predict(X_test)

Kernel Matrix:
[[1.         0.60653066 0.011109  ]
 [0.60653066 1.         0.13533528]
 [0.011109   0.13533528 1.        ]]
Dual coefficients (alpha):
[0.24193835 1.00323689 2.57435931]


Test Kernel:
[[0.32465247 0.8824969  0.32465247]]


array([1.79967143])